# 01 — Prepare Data (Colab → Google Drive)

Chạy tiền xử lý ChatML cho Task 1/2/3 trên Colab và **ghi output lên Google Drive**.

## Workflow khuyến nghị (KHÔNG cần push mỗi lần sửa)

1. Giữ **source code trên Drive** (cùng project với `data/`).
2. Colab import trực tiếp từ Drive — sửa file rồi **re-run cell load source** (tự clear cache `src.*`), không cần git.
3. Chỉ `git push` khi code đã ổn định (milestone), không dùng GitHub làm hot-reload.

```text
LLM/Final Project/
├── llm-final-project/     ← SOURCE CODE (chạy từ đây)
│   ├── src/
│   ├── scripts/
│   └── colab/
└── data/                  ← DATA
    ├── raw/sample_dataset/...
    ├── processed/
    ├── splits/
    └── outputs/
```


## 0. Cau hinh tu `.env` (moi nguoi mot branch)

Copy `.env.example` -> `.env` (local hoac tren Drive). **Khong commit `.env`.**

Bien quan trong: `REPO_URL`, `REPO_BRANCH`, `DRIVE_DATA_DIR`, `REPO_DIR_ON_DRIVE`.


In [37]:
# Defaults — override bang .env (Drive hoac repo)
from pathlib import Path
import os

DRIVE_ENV_CANDIDATES = [
    "/content/drive/MyDrive/LLM/Final Project/.env",
    "/content/drive/MyDrive/LLM/Final Project/llm-final-project/.env",
]

os.environ.setdefault("REPO_URL", "https://github.com/tiendinhquang2104/llm-final-project.git")
os.environ.setdefault("REPO_BRANCH", "dev/process_data")
os.environ.setdefault("DRIVE_DATA_DIR", "/content/drive/MyDrive/LLM/Final Project/data")
os.environ.setdefault("REPO_DIR_ON_DRIVE", "/content/drive/MyDrive/LLM/Final Project/llm-final-project")
os.environ.setdefault("RANDOM_SEED", "42")
os.environ.setdefault("VAL_RATIO", "0.2")

REPO_URL = os.environ["REPO_URL"]
REPO_BRANCH = os.environ["REPO_BRANCH"]
DRIVE_DATA_DIR = os.environ["DRIVE_DATA_DIR"]
REPO_DIR_ON_DRIVE = os.environ["REPO_DIR_ON_DRIVE"]
SEED = int(os.environ["RANDOM_SEED"])
VAL_RATIO = float(os.environ["VAL_RATIO"])

print("REPO_BRANCH =", REPO_BRANCH)
print("DRIVE_DATA_DIR =", DRIVE_DATA_DIR)
print("(Sau mount Drive, cell tiep theo se reload .env rieng neu co)")


REPO_BRANCH = dev/process_data
DRIVE_DATA_DIR = /content/drive/MyDrive/LLM/Final Project/data
(Sau mount Drive, cell tiep theo se reload .env rieng neu co)


## 1. Mount Google Drive & resolve folder

In [38]:
from pathlib import Path
import os, json, subprocess, sys

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Drive .env = source of truth cho may nay
for env_path in DRIVE_ENV_CANDIDATES:
    ep = Path(env_path)
    if not ep.is_file():
        continue
    for line in ep.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        k, v = k.strip(), v.strip().strip('"').strip("'")
        os.environ[k] = v
    print("Loaded Drive env:", ep)
    break

REPO_URL = os.environ.get("REPO_URL", REPO_URL)
REPO_BRANCH = os.environ.get("REPO_BRANCH", REPO_BRANCH)
DRIVE_DATA_DIR = os.environ.get("DRIVE_DATA_DIR", DRIVE_DATA_DIR)
REPO_DIR_ON_DRIVE = os.environ.get("REPO_DIR_ON_DRIVE", REPO_DIR_ON_DRIVE)
SEED = int(os.environ.get("RANDOM_SEED", SEED))
VAL_RATIO = float(os.environ.get("VAL_RATIO", VAL_RATIO))


def resolve_drive_data(preferred: str) -> Path:
    # Chi dung path da mount — khong goi auth.authenticate_user()
    pref = Path(preferred)
    if pref.exists():
        return pref.resolve()

    found = []
    roots = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]
    for root in roots:
        if not root.exists():
            continue
        for d in root.rglob("data"):
            if not d.is_dir():
                continue
            try:
                rel = d.relative_to(root)
            except ValueError:
                continue
            if len(rel.parts) > 3:
                continue
            found.append(d)
            if len(found) >= 20:
                break

    for d in found:
        if (d / "raw").exists() or (d / "exams.json").exists() or (d / "processed").exists():
            return d.resolve()
    if found:
        print("Ung cu vien data tren Drive:")
        for d in found:
            print(" -", d)
        return found[0].resolve()

    pref.mkdir(parents=True, exist_ok=True)
    print("Da tao folder moi:", pref)
    print("Neu sai: sua DRIVE_DATA_DIR trong .env tren Drive, roi chay lai.")
    return pref.resolve()


DRIVE_DATA = resolve_drive_data(DRIVE_DATA_DIR)
RAW_DIR = DRIVE_DATA / "raw"
PROCESSED_DIR = DRIVE_DATA / "processed"
SPLITS_DIR = DRIVE_DATA / "splits"
OUTPUTS_DIR = DRIVE_DATA / "outputs"
DRIVE_FOLDER_ID = "1P31-GtTh5GHcUxO7TJhftSONuBTvvKwI"

for d in (RAW_DIR, PROCESSED_DIR, SPLITS_DIR, OUTPUTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("REPO_BRANCH   =", REPO_BRANCH)
print("DRIVE_DATA   =", DRIVE_DATA)
print("RAW_DIR      =", RAW_DIR)
print("PROCESSED_DIR=", PROCESSED_DIR)
print("exists?      =", DRIVE_DATA.exists())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
REPO_BRANCH   = dev/process_data
DRIVE_DATA   = /content/drive/MyDrive/LLM/Final Project/data
RAW_DIR      = /content/drive/MyDrive/LLM/Final Project/data/raw
PROCESSED_DIR= /content/drive/MyDrive/LLM/Final Project/data/processed
exists?      = True


## 2. Load source code từ Drive (không clone mỗi lần)

Ưu tiên chạy code **trực tiếp từ Drive**.
Sau khi sync `.py` mới: **re-run cell này** (có clear `sys.modules` cho package `src`) rồi chạy tiếp các cell dưới.
Không dùng `%autoreload` — trên Colab Python 3.13 bị lỗi `No module named 'imp'`.


In [39]:
from pathlib import Path
import os, sys, subprocess

def ensure_repo() -> Path:
    drive_repo = Path(REPO_DIR_ON_DRIVE)
    marker = drive_repo / "scripts" / "prepare_data.py"

    if marker.exists():
        git_dir = drive_repo / ".git"
        if git_dir.exists() and REPO_BRANCH:
            try:
                subprocess.check_call(
                    ["git", "-C", str(drive_repo), "fetch", "--depth", "1", "origin", REPO_BRANCH]
                )
                subprocess.check_call(["git", "-C", str(drive_repo), "checkout", REPO_BRANCH])
                subprocess.check_call(
                    ["git", "-C", str(drive_repo), "pull", "--ff-only", "origin", REPO_BRANCH]
                )
                print("Updated Drive repo -> branch", REPO_BRANCH)
            except subprocess.CalledProcessError as e:
                print("[warn] khong cap nhat duoc branch:", e, "; dung code dang co tren Drive")
        return drive_repo

    if REPO_URL:
        drive_repo.parent.mkdir(parents=True, exist_ok=True)
        if drive_repo.exists():
            raise FileNotFoundError(
                f"{drive_repo} ton tai nhung thieu scripts/prepare_data.py. "
                "Xoa folder do hoac sua REPO_DIR_ON_DRIVE trong .env."
            )
        cmd = [
            "git", "clone", "--branch", REPO_BRANCH, "--single-branch",
            REPO_URL, str(drive_repo),
        ]
        print("Bootstrap:", " ".join(cmd))
        subprocess.check_call(cmd)
        return drive_repo

    raise FileNotFoundError(
        "Chua co source tren Drive.\n"
        f"Upload/sync repo vao: {REPO_DIR_ON_DRIVE}\n"
        "Hoac set REPO_URL + REPO_BRANCH trong .env de clone 1 lan."
    )

LOCAL_REPO = ensure_repo()
os.chdir(LOCAL_REPO)
sys.path = [str(LOCAL_REPO)] + [x for x in sys.path if x != str(LOCAL_REPO)]

for mod in list(sys.modules):
    if mod == "src" or mod.startswith("src."):
        del sys.modules[mod]

print("Using source:", LOCAL_REPO)
print("REPO_BRANCH :", REPO_BRANCH)
print("prepare_data.py exists:", (LOCAL_REPO / "scripts" / "prepare_data.py").exists())


Updated Drive repo -> branch dev/process_data
Using source: /content/drive/MyDrive/LLM/Final Project/llm-final-project
REPO_BRANCH : dev/process_data
prepare_data.py exists: True


## 3. Kiểm tra raw dataset trên Drive

Layout hợp lệ (giống local):
- `data/raw/sample_dataset/sample_dataset/exams.json`
- `data/raw/sample_dataset/exams.json`
- `data/raw/exams.json`


In [40]:
from src.data.loader import resolve_dataset_root

print("RAW_DIR contents:")
if RAW_DIR.exists():
    for child in sorted(RAW_DIR.iterdir()):
        print(" -", child.name, "(dir)" if child.is_dir() else "(file)")
else:
    print(" (missing)")

try:
    DATA_DIR = resolve_dataset_root(RAW_DIR)
    print("✓ Raw dataset OK:", DATA_DIR)
    print("  files:", sorted(p.name for p in DATA_DIR.iterdir() if p.is_file())[:10])
except FileNotFoundError as e:
    print("✗", e)
    print()
    print("Hãy upload dataset vào:", RAW_DIR)
    print("Layout hợp lệ ví dụ:")
    print("  raw/sample_dataset/sample_dataset/exams.json")
    print("  raw/sample_dataset/exams.json")
    print("  raw/exams.json")
    print("Link folder:", f"https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
    raise


RAW_DIR contents:
 - sample_dataset (dir)
✓ Raw dataset OK: /content/drive/MyDrive/LLM/Final Project/data/raw/sample_dataset/sample_dataset
  files: ['.DS_Store', 'README.md', 'exams.json', 'label_space.json', 'task1_grading.json', 'task2_error_taxonomy.json', 'task3_feedback.json']


## 4. Chạy preprocess → ghi lên Drive

In [41]:
from src.data.preprocess import build_unified_jsonl

# Ghi processed vào Drive; splits cũng sẽ vào Drive data/splits vì output_dir tên 'processed' và parent là 'data'
paths = build_unified_jsonl(
    data_dir=DATA_DIR,
    output_dir=PROCESSED_DIR,
    splits_dir=SPLITS_DIR,
    val_ratio=VAL_RATIO,
    seed=SEED,
    check_leakage=True,
)

print("=== Written ===")
for k, p in paths.items():
    size = p.stat().st_size if p.exists() else 0
    print(f"{k:18s} {p}  ({size:,} bytes)")

manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"))
print("\nmanifest:", json.dumps(manifest, ensure_ascii=False, indent=2))

=== Written ===
unified_samples    /content/drive/MyDrive/LLM/Final Project/data/processed/unified_samples.jsonl  (149,749 bytes)
train_unified      /content/drive/MyDrive/LLM/Final Project/data/processed/train_unified.jsonl  (804,327 bytes)
val_unified        /content/drive/MyDrive/LLM/Final Project/data/processed/val_unified.jsonl  (190,391 bytes)
task1_train        /content/drive/MyDrive/LLM/Final Project/data/processed/task1_train.jsonl  (271,947 bytes)
task1_val          /content/drive/MyDrive/LLM/Final Project/data/processed/task1_val.jsonl  (64,514 bytes)
task2_train        /content/drive/MyDrive/LLM/Final Project/data/processed/task2_train.jsonl  (259,426 bytes)
task2_val          /content/drive/MyDrive/LLM/Final Project/data/processed/task2_val.jsonl  (61,538 bytes)
task3_train        /content/drive/MyDrive/LLM/Final Project/data/processed/task3_train.jsonl  (272,954 bytes)
task3_val          /content/drive/MyDrive/LLM/Final Project/data/processed/task3_val.jsonl  (64,339 byte

## 5. EDA summary → Drive `data/outputs/`

In [42]:
import subprocess

eda_out = OUTPUTS_DIR / "eda_summary.json"
subprocess.check_call([
    sys.executable, "scripts/eda_report.py",
    "--data_dir", str(DATA_DIR),
    "--out", str(eda_out),
])
print(eda_out.read_text(encoding="utf-8")[:1500])

{
  "n_samples": 32,
  "exam_type": {
    "multi_problem": 15,
    "single_problem": 17
  },
  "total_score_dist": {
    "0": 3,
    "1": 5,
    "2": 4,
    "3": 1,
    "6": 1,
    "7": 7,
    "8": 2,
    "9": 4,
    "10": 5
  },
  "rubric_dist": {
    "compilable": {
      "1": 26,
      "0": 6
    },
    "io_format": {
      "1": 21,
      "0": 11
    },
    "logic": {
      "2": 9,
      "4": 7,
      "0": 13,
      "3": 2,
      "1": 1
    },
    "edge_case": {
      "2": 10,
      "1": 9,
      "0": 13
    },
    "complexity": {
      "1": 20,
      "0": 12
    },
    "code_quality": {
      "1": 24,
      "0": 8
    }
  },
  "taxonomy_empty": 4,
  "taxonomy_counts": {
    "Lỗi biên dịch": 6,
    "Lỗi nhập/xuất": 2,
    "Lỗi logic": 12,
    "Lỗi vòng lặp": 6,
    "Lỗi mảng/chuỗi": 1,
    "Lỗi hàm": 5,
    "Lỗi edge case": 0,
    "Lỗi thuật toán": 4,
    "Lỗi hard-code": 3,
    "Lỗi style": 2
  },
  "feedback_levels": {
    "Level 2 - Chẩn đoán": 18,
    "Level 1 - Gợi ý nhẹ": 4,
 

## 6. Smoke-check + đường dẫn dùng cho training sau này

In [43]:
train = PROCESSED_DIR / "train_unified.jsonl"
val = PROCESSED_DIR / "val_unified.jsonl"
assert train.exists() and val.exists()

n_train = sum(1 for _ in train.open(encoding="utf-8"))
n_val = sum(1 for _ in val.open(encoding="utf-8"))
row = json.loads(train.open(encoding="utf-8").readline())
assert "<|im_start|>" in row["text"]
assert row["task"] in {"task1", "task2", "task3"}

print(f"OK train={n_train} val={n_val}")
print()
print("Dùng các path này ở notebook train:")
print("  TRAIN_JSONL =", train)
print("  VAL_JSONL   =", val)
print("  SPLITS_DIR  =", SPLITS_DIR)
print("  DRIVE_DATA  =", DRIVE_DATA)
print()
print("Mở Drive:", f"https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")

OK train=78 val=18

Dùng các path này ở notebook train:
  TRAIN_JSONL = /content/drive/MyDrive/LLM/Final Project/data/processed/train_unified.jsonl
  VAL_JSONL   = /content/drive/MyDrive/LLM/Final Project/data/processed/val_unified.jsonl
  SPLITS_DIR  = /content/drive/MyDrive/LLM/Final Project/data/splits
  DRIVE_DATA  = /content/drive/MyDrive/LLM/Final Project/data

Mở Drive: https://drive.google.com/drive/folders/1P31-GtTh5GHcUxO7TJhftSONuBTvvKwI
